# Mastercard Data Quest 2026
## Hidden Entrepreneur Detection — ML Pipeline

**Task:** Identify consumer cardholders who exhibit business-like transaction behaviour using MasterCard transaction data.

| | |
|---|---|
| **Dataset** | 25,000 business cards · 80,000 consumer cards · 12M transactions |
| **Model** | LightGBM + Optuna hyperparameter tuning |
| **Output** | Ranked list of 80,000 consumer cards by `p_business` score |

## 0. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    f1_score, roc_curve, precision_recall_curve,
)

# Paths — relative to notebook location, works on any OS
BASE_DIR   = Path(".")
DATA_DIR   = BASE_DIR / "data" / "raw"
OUT_DIR    = BASE_DIR / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BIZ_PATH   = DATA_DIR / "business_cards_MDQ.parquet"
CONS_PATH  = DATA_DIR / "consumer_cards_MDQ.parquet"
MERCH_PATH = DATA_DIR / "merchants_reference.parquet"

RANDOM_STATE = 42
print("✓ Imports complete")

## 1. Data Loading

Three parquet files:
- `business_cards_MDQ.parquet` — known business cardholders (label = 1)
- `consumer_cards_MDQ.parquet` — consumer cardholders (label = 0, target for scoring)
- `merchants_reference.parquet` — MCC codes and merchant metadata

In [ ]:
for path in [BIZ_PATH, CONS_PATH, MERCH_PATH]:
    if not path.exists():
        print(f"[ERROR] File not found: {path}")
        sys.exit(1)

df_biz   = pd.read_parquet(BIZ_PATH);   df_biz["segment"]  = 1
df_cons  = pd.read_parquet(CONS_PATH);  df_cons["segment"] = 0
df_merch = pd.read_parquet(MERCH_PATH)

print(f"Business cards:  {len(df_biz):>10,} transactions | {df_biz['card_number'].nunique():>7,} cards")
print(f"Consumer cards:  {len(df_cons):>10,} transactions | {df_cons['card_number'].nunique():>7,} cards")
print(f"Merchants ref:   {len(df_merch):>10,} merchants")

## 2. Preprocessing

- Merge business + consumer into one dataframe
- Memory optimisation: downcast float64 → float32, int64 → int32
- Parse timestamps, extract hour / weekday / month / week
- Join merchants reference to get `merchant_country` and `recurring_capable`

In [ ]:
df = pd.concat([df_biz, df_cons], ignore_index=True)

for col in df.select_dtypes("float64").columns:
    df[col] = df[col].astype("float32")
for col in df.select_dtypes("int64").columns:
    df[col] = df[col].astype("int32")

df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
if "transaction_timestamp" in df.columns:
    df["transaction_timestamp"] = pd.to_datetime(df["transaction_timestamp"], errors="coerce")
    df["hour"]    = df["transaction_timestamp"].dt.hour
    df["weekday"] = df["transaction_timestamp"].dt.dayofweek
else:
    df["hour"]    = 12
    df["weekday"] = df["transaction_date"].dt.dayofweek

df["month"] = df["transaction_date"].dt.to_period("M")
df["week"]  = df["transaction_timestamp"].dt.isocalendar().week.astype("int32")

df = df.merge(
    df_merch[["merchant_id", "mcc", "merchant_country", "recurring_capable"]],
    on="merchant_id", how="left", suffixes=("", "_merch")
)
if "mcc_merch" in df.columns:
    df["mcc"] = df["mcc_merch"].fillna(df["mcc"])
    df.drop(columns=["mcc_merch"], inplace=True)

print(f"Combined dataset: {len(df):,} transactions | {df['card_number'].nunique():,} unique cards")
print(f"Columns: {list(df.columns)}")

## 3. Exploratory Data Analysis

Six-panel comparison: Business vs Consumer across key dimensions.

In [ ]:
amount_col  = "transaction_amount_kzt" if "transaction_amount_kzt" in df.columns else               next(c for c in df.columns if "amount" in c.lower())
seg_labels  = {1: "Business", 0: "Consumer"}
colors      = {1: "#2196F3", 0: "#FF9800"}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("EDA: Business vs Consumer — Key Distributions", fontsize=14, fontweight="bold")

# (A) Transactions per card
txn_per_card = df.groupby(["card_number", "segment"]).size().reset_index(name="n")
for seg, grp in txn_per_card.groupby("segment"):
    axes[0,0].hist(grp["n"].clip(upper=grp["n"].quantile(0.99)),
                   bins=50, alpha=0.6, label=seg_labels[seg],
                   color=colors[seg], density=True)
axes[0,0].set_title("Transactions per card (up to 99th pctile)")
axes[0,0].legend()

# (B) log(avg amount)
avg_amt = df.groupby(["card_number","segment"])[amount_col].mean().reset_index(name="avg")
for seg, grp in avg_amt.groupby("segment"):
    axes[0,1].hist(np.log1p(grp["avg"].clip(lower=0)),
                   bins=50, alpha=0.6, label=seg_labels[seg],
                   color=colors[seg], density=True)
axes[0,1].set_title("log(Avg transaction amount + 1)")
axes[0,1].legend()

# (C) Online vs POS
if "channel" in df.columns:
    ch = df.groupby(["segment","channel"]).size().unstack(fill_value=0)
    ch.index = [seg_labels[i] for i in ch.index]
    ch.plot(kind="bar", ax=axes[0,2], color=["#4CAF50","#F44336"])
    axes[0,2].set_title("Online vs POS transactions")
    axes[0,2].tick_params(axis="x", rotation=0)

# (D) Top-15 MCC — Business
top_mcc = df[df["segment"]==1]["mcc"].value_counts().head(15)
axes[1,0].barh(top_mcc.index.astype(str), top_mcc.values, color="#2196F3")
axes[1,0].set_title("Top-15 MCC codes (Business)")
axes[1,0].invert_yaxis()

# (E) By hour
hour_d = df.groupby(["hour","segment"]).size().reset_index(name="cnt")
for seg, grp in hour_d.groupby("segment"):
    axes[1,1].plot(grp["hour"], grp["cnt"], label=seg_labels[seg], color=colors[seg])
axes[1,1].set_title("Transactions by hour of day")
axes[1,1].set_xlabel("Hour")
axes[1,1].legend()

# (F) By weekday
wd_names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
wd_d = df.groupby(["weekday","segment"]).size().reset_index(name="cnt")
for seg, grp in wd_d.groupby("segment"):
    axes[1,2].plot(grp["weekday"], grp["cnt"], marker="o",
                   label=seg_labels[seg], color=colors[seg])
axes[1,2].set_xticks(range(7)); axes[1,2].set_xticklabels(wd_names)
axes[1,2].set_title("Transactions by day of week")
axes[1,2].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ EDA saved → reports/eda_overview.png")

## 4. Feature Engineering

### Business MCC Set
Data-driven approach: MCC codes that appear **3× more frequently** in business transactions than consumer transactions are flagged as business-characteristic.

### Card-level Features
All transactions are aggregated to card level — one row per card, ~22 behavioral features.

| Feature Group | Key Features |
|---|---|
| Volume | `total_txn_count`, `avg_amount`, `amount_cv` |
| Timing | `biz_hours_share`, `weekend_share`, `night_share` |
| Channel | `online_share` |
| MCC Profile | `biz_mcc_share`, `mcc_entropy`, `mcc_hhi` |
| Merchants | `unique_merchants`, `hhi_merchants` |
| Recurring | `recurring_share`, `biz_recurring_share` |

In [ ]:
# Business MCC Set — data-driven, population level (no leakage)
mcc_biz   = df[df["segment"]==1]["mcc"].value_counts(normalize=True)
mcc_cons  = df[df["segment"]==0]["mcc"].value_counts(normalize=True)
mcc_ratio = (mcc_biz / (mcc_cons + 1e-9)).fillna(0)
BIZ_MCC_SET = set(mcc_ratio[mcc_ratio >= 3].index)
print(f"BIZ_MCC_SET: {len(BIZ_MCC_SET)} codes (appear 3x+ more in business cards)")

# Binary flags
df = df.copy()
df["is_online"] = (df["channel"].str.lower()=="online").astype("int8")                    if "channel" in df.columns else 0

recurring_col = next((c for c in df.columns if c.lower()=="is_recurring"), None)
df["is_recurring_flag"] = df[recurring_col].astype("int8") if recurring_col else 0

if "merchant_country" in df.columns:
    df["is_foreign_merchant"] = (df["merchant_country"] != "Kazakhstan").astype("int8")
elif "country" in df.columns:
    df["is_foreign_merchant"] = (df["country"] != "Kazakhstan").astype("int8")
else:
    df["is_foreign_merchant"] = 0

df["is_biz_hours"] = ((df["hour"]>=9)&(df["hour"]<18)&(df["weekday"]<5)).astype("int8")
df["is_weekend"]   = (df["weekday"]>=5).astype("int8")
df["is_night"]     = ((df["hour"]<6)|(df["hour"]>=22)).astype("int8")
df["is_biz_mcc"]   = df["mcc"].isin(BIZ_MCC_SET).astype("int8")
df["recurring_capable"] = df["recurring_capable"].fillna(0).astype("int8")

# Entropy and HHI helpers
from scipy.stats import entropy as scipy_entropy
def _entropy(s):
    vc = s.value_counts(normalize=True)
    return float(-(vc * np.log(vc+1e-9)).sum())
def _hhi(s):
    vc = s.value_counts(normalize=True)
    return float((vc**2).sum())

# Aggregation
card_feats = df.groupby("card_number").agg(
    segment                 =(amount_col,            "first"),
    total_txn_count         =(amount_col,            "count"),
    total_amount            =(amount_col,            "sum"),
    avg_amount              =(amount_col,            "mean"),
    median_amount           =(amount_col,            "median"),
    std_amount              =(amount_col,            "std"),
    max_amount              =(amount_col,            "max"),
    biz_hours_share         =("is_biz_hours",        "mean"),
    weekend_share           =("is_weekend",          "mean"),
    night_share             =("is_night",            "mean"),
    online_share            =("is_online",           "mean"),
    foreign_merchant_share  =("is_foreign_merchant", "mean"),
    recurring_share         =("is_recurring_flag",   "mean"),
    recurring_capable_share =("recurring_capable",   "mean"),
    unique_mcc              =("mcc",                 "nunique"),
    biz_mcc_share           =("is_biz_mcc",          "mean"),
    unique_merchants        =("merchant_id",         "nunique"),
    active_months           =("month",               "nunique"),
).reset_index()

# Fix segment column (was aggregated as "first" of amount — redo)
seg_map = df.groupby("card_number")["segment"].first()
card_feats["segment"] = card_feats["card_number"].map(seg_map)

card_feats["mcc_entropy"]   = df.groupby("card_number")["mcc"].apply(_entropy).values
card_feats["hhi_merchants"] = df.groupby("card_number")["merchant_id"].apply(_hhi).values

months = card_feats["active_months"].clip(lower=1)
card_feats["txn_per_month"]    = card_feats["total_txn_count"] / months
card_feats["amount_per_month"] = card_feats["total_amount"]    / months
card_feats["merch_per_month"]  = card_feats["unique_merchants"] / months
card_feats["amount_cv"]        = card_feats["std_amount"] / (card_feats["avg_amount"]+1e-9)
card_feats.fillna(0, inplace=True)

FEATURE_COLS = [c for c in card_feats.columns if c not in ("card_number","segment")]
print(f"✓ Features built: {len(card_feats):,} cards × {len(FEATURE_COLS)} features")
card_feats.head(3)

## 5. Model Training

Three models compared:
1. **Logistic Regression** — linear baseline
2. **Random Forest** — non-linear baseline
3. **LightGBM + Optuna** — final model (30-trial Bayesian hyperparameter search with 3-fold CV)

In [ ]:
X = card_feats[FEATURE_COLS].values.astype("float32")
y = card_feats["segment"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train):,} cards | Test: {len(X_test):,} cards | Business rate: {y_train.mean():.3f}")

In [ ]:
results = {}

# 1. Logistic Regression
print("[1/3] Logistic Regression...")
lr = Pipeline([("sc", StandardScaler()),
               ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                                          C=0.1, random_state=RANDOM_STATE))])
lr.fit(X_train, y_train)
lr_proba = lr.predict_proba(X_test)[:,1]
results["LogisticRegression"] = {"proba": lr_proba,
    "roc_auc": roc_auc_score(y_test, lr_proba),
    "pr_auc":  average_precision_score(y_test, lr_proba)}
print(f"  ROC-AUC={results['LogisticRegression']['roc_auc']:.4f} | "
      f"PR-AUC={results['LogisticRegression']['pr_auc']:.4f}")

# 2. Random Forest
print("[2/3] Random Forest...")
rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                             class_weight="balanced",
                             random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:,1]
results["RandomForest"] = {"proba": rf_proba,
    "roc_auc": roc_auc_score(y_test, rf_proba),
    "pr_auc":  average_precision_score(y_test, rf_proba)}
print(f"  ROC-AUC={results['RandomForest']['roc_auc']:.4f} | "
      f"PR-AUC={results['RandomForest']['pr_auc']:.4f}")

# 3. LightGBM + Optuna
print("[3/3] LightGBM + Optuna (30 trials)...")
def objective(trial):
    params = {
        "verbosity": -1,
        "n_estimators":      trial.suggest_int("n_estimators", 200, 600),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 150),
        "max_depth":         trial.suggest_int("max_depth", 4, 12),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "class_weight": "balanced", "random_state": RANDOM_STATE, "n_jobs": -1,
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    return cross_val_score(lgb.LGBMClassifier(**params), X_train, y_train,
                           cv=skf, scoring="roc_auc", n_jobs=-1).mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)
print(f"  Best CV AUC: {study.best_value:.4f}")

best_p = {**study.best_params,
          "verbosity":-1, "class_weight":"balanced",
          "random_state":RANDOM_STATE, "n_jobs":-1}
lgb_clf = lgb.LGBMClassifier(**best_p)
lgb_clf.fit(X_train, y_train)
lgb_proba = lgb_clf.predict_proba(X_test)[:,1]
results["LightGBM"] = {"proba": lgb_proba,
    "roc_auc": roc_auc_score(y_test, lgb_proba),
    "pr_auc":  average_precision_score(y_test, lgb_proba)}
print(f"  ROC-AUC={results['LightGBM']['roc_auc']:.4f} | "
      f"PR-AUC={results['LightGBM']['pr_auc']:.4f}")

## 6. Evaluation & Metrics

- ROC-AUC and PR-AUC for all three models
- Optimal classification threshold selected by maximising F1
- Confusion matrix at optimal threshold

In [ ]:
# Model comparison table
pd.DataFrame(results).T[["roc_auc","pr_auc"]].round(4).rename(
    columns={"roc_auc":"ROC-AUC","pr_auc":"PR-AUC"})

In [ ]:
lgb_proba = results["LightGBM"]["proba"]
prec, rec, thresholds = precision_recall_curve(y_test, lgb_proba)
f1_scores = 2*prec*rec/(prec+rec+1e-9)
best_threshold = float(thresholds[np.argmax(f1_scores[:-1])])
lgb_pred = (lgb_proba >= best_threshold).astype(int)
print(f"Optimal threshold (max F1): {best_threshold:.3f}")

cm = confusion_matrix(y_test, lgb_pred)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Model Evaluation — LightGBM (best model)", fontsize=13, fontweight="bold")

# ROC curves
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["proba"])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})")
axes[0].plot([0,1],[0,1],"k--")
axes[0].set_title("ROC Curves"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend()

# Confusion Matrix
axes[1].imshow(cm, cmap="Blues")
axes[1].set_title(f"Confusion Matrix (threshold={best_threshold:.2f})")
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(["Consumer","Business"])
axes[1].set_yticklabels(["Consumer","Business"])
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                     color="white" if cm[i,j]>cm.max()/2 else "black",
                     fontsize=14, fontweight="bold")

# PR curves
for name, res in results.items():
    p_c, r_c, _ = precision_recall_curve(y_test, res["proba"])
    axes[2].plot(r_c, p_c, label=f"{name} (AP={res['pr_auc']:.3f})")
axes[2].set_title("Precision-Recall Curves")
axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
axes[2].legend()

plt.tight_layout()
plt.savefig(OUT_DIR/"model_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nClassification Report (LightGBM):")
print(classification_report(y_test, lgb_pred, target_names=["Consumer","Business"]))

## 7. SHAP — Model Explainability

SHAP (SHapley Additive exPlanations) shows **which features push a card's score up or down**, making the model auditable.

- **Bar chart**: mean absolute SHAP — overall feature importance
- **Beeswarm**: direction and magnitude per sample

In [ ]:
rng         = np.random.default_rng(RANDOM_STATE)
idx         = rng.choice(len(X_test), size=min(2000,len(X_test)), replace=False)
X_sample    = pd.DataFrame(X_test[idx], columns=FEATURE_COLS)

explainer   = shap.TreeExplainer(lgb_clf)
shap_values = explainer.shap_values(X_sample)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# Bar
plt.figure(figsize=(10,7))
shap.summary_plot(sv, X_sample, max_display=20, show=False, plot_type="bar")
plt.title("SHAP Feature Importance (LightGBM)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR/"shap_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Beeswarm
plt.figure(figsize=(10,10))
shap.summary_plot(sv, X_sample, max_display=20, show=False)
plt.title("SHAP Summary — Feature Direction & Magnitude", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR/"shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ SHAP plots saved")

## 8. Scoring Consumer Cards

All 80,000 consumer cards are scored. Output: `p_business` — probability of hidden business behaviour (0 to 1).

In [ ]:
consumer_cards = card_feats[card_feats["segment"]==0].copy()
X_consumer     = consumer_cards[FEATURE_COLS].values.astype("float32")
consumer_cards["p_business"] = lgb_clf.predict_proba(X_consumer)[:,1].round(6)

submission = consumer_cards[["card_number","p_business"]].sort_values("p_business", ascending=False)
submission.to_csv(OUT_DIR/"final_submission.csv", index=False)
print(f"✓ Scored {len(submission):,} consumer cards")
print(f"  Score > 0.75 (hot):  {(consumer_cards['p_business']>=0.75).sum():,} cards")
print(f"  Score > 0.50 (warm): {(consumer_cards['p_business']>=0.50).sum():,} cards")
submission.head(10)

## 9. Top-50 Hidden Entrepreneur Profiles

Detailed feature breakdown for the top 50 highest-scored consumer cards.

In [ ]:
top50 = consumer_cards.nlargest(50, "p_business")
top50.to_csv(OUT_DIR/"top_50_candidates_detailed.csv", index=False)

show_cols = ["card_number","p_business","total_txn_count",
             "avg_amount","biz_mcc_share","biz_hours_share","online_share","mcc_entropy"]
top50[show_cols].head(10).round(4)

## 10. Segment Profiling

Three groups compared:
- **Real Business** — known business cards (ground truth)
- **Hidden Entrepreneur** — consumer cards with `p_business ≥ threshold`
- **Regular Consumer** — remaining consumer cards

In [ ]:
consumer_cards["group"] = np.where(
    consumer_cards["p_business"] >= best_threshold,
    "Hidden Entrepreneur", "Regular Consumer"
)
biz_prof = card_feats[card_feats["segment"]==1][FEATURE_COLS].assign(group="Real Business")
groups   = pd.concat([consumer_cards[FEATURE_COLS+["group"]], biz_prof])

profile_cols = ["total_txn_count","avg_amount","unique_merchants",
                "biz_mcc_share","biz_hours_share","recurring_share","online_share"]

summary = groups.groupby("group")[profile_cols].median().round(3)
print("Median values by segment:")
display(summary)

group_colors = {"Real Business":"#2196F3","Hidden Entrepreneur":"#FF5722","Regular Consumer":"#4CAF50"}
fig, axes = plt.subplots(2, 4, figsize=(20,10))
fig.suptitle("Segment Profiles: Real Business / Hidden Entrepreneur / Regular Consumer",
             fontsize=13, fontweight="bold")
axes = axes.flatten()
for i, col in enumerate(profile_cols):
    for grp, color in group_colors.items():
        data = groups[groups["group"]==grp][col].clip(upper=groups[col].quantile(0.99))
        axes[i].hist(data, bins=40, alpha=0.5, label=grp, color=color, density=True)
    axes[i].set_title(col); axes[i].legend(fontsize=7)
cnt = groups["group"].value_counts()
axes[7].bar(cnt.index, cnt.values, color=[group_colors[g] for g in cnt.index])
axes[7].set_title("Segment Sizes")
for j,(g,v) in enumerate(cnt.items()):
    axes[7].text(j, v+cnt.max()*0.01, f"{v:,}", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR/"segment_profiles.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

| Output file | Description |
|---|---|
| `reports/eda_overview.png` | 6-panel EDA comparison |
| `reports/model_evaluation.png` | ROC, PR curves, Confusion Matrix |
| `reports/shap_feature_importance.png` | Top-20 features by SHAP importance |
| `reports/shap_summary.png` | SHAP beeswarm — direction of influence |
| `reports/segment_profiles.png` | Segment comparison distributions |
| `reports/final_submission.csv` | All 80,000 consumer cards ranked by `p_business` |
| `reports/top_50_candidates_detailed.csv` | Top 50 hidden entrepreneur profiles |